<a href="https://colab.research.google.com/github/T2718/Project/blob/main/ec%E3%81%AE%E3%83%AA%E3%82%B9%E3%83%88%E6%93%8D%E4%BD%9Cv1-4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title 初期設定

from google.colab import drive
import pathlib
from bs4 import BeautifulSoup
import re

drive.mount('/content/drive')

importDriver = True #@param {type: "boolean"}

driver = None

if importDriver:
  !pip install selenium
  !pip install webdriver-manager
  !apt-get purge chromium-browser chromium-chromedriver -y
  !wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
  !echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' | tee /etc/apt/sources.list.d/google-chrome.list
  !apt-get update
  !apt-get install google-chrome-stable -y
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def initDriver():
  global driver
  chrome_options = Options()
  chrome_options.add_argument('--headless')
  chrome_options.add_argument('--no-sandbox')
  chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
  )
  chrome_options.add_argument("--disable-blink-features=AutomationControlled")
  chrome_options.add_argument('--disable-dev-shm-usage')
  chrome_options.binary_location = '/usr/bin/google-chrome'

  # Automatically download and use the correct chromedriver
  service = Service(ChromeDriverManager().install())
  driver = webdriver.Chrome(service=service, options=chrome_options)


folder = "/content/drive/MyDrive/共有フォルダ/情報/ec" #@param {"type": "string"}
folder_path = pathlib.Path(folder)

url_list_path = folder_path / 'url-list.txt'
url_sort_list_path = folder_path / 'url-sort-list.txt'
spank_list_path = folder_path / 'spank-list.txt'
info_list_path = folder_path / 'info-list.txt'

def test(text, kind, option={}):
  soup = BeautifulSoup(text, 'html.parser')

  if kind == 'extractAllCode':
    for element in soup.find_all(string=True):
      element.extract()
    print(soup.prettify())
  elif kind == 'allCode':
    print(soup.prettify())
  elif kind == 'title':
    element = soup.find('meta', attrs={'name': 'twitter:title'})
    print(element['content'])
  elif kind == 'information':
    information = soup.find('div', attrs={'class': 'information_box'})
    print(information)
  elif kind == 'video':
    video = soup.find(id="video")
    print(video)
  elif kind == "search":
    for find in re.finditer(option['re'], text):
      index = find.start()
      range = option['range'] if 'range' in option.keys() else 300
      print(text[index-range:index+range])
      print("\n-----------------------------–\n")



Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 12.1 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'chromium-browser' is not installed, so not removed
Package 'chromium-chromedriver' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
OK
deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hi

In [5]:
#@title MP4ダウンロードヘルパー関数

import requests
from urllib.parse import urlparse
import pathlib

def download_mp4(video_url, filename=None):
    """URLからMP4ファイルをダウンロードするヘルパー関数。"""
    if not filename:
        parsed_url = urlparse(video_url)
        filename = pathlib.Path(parsed_url.path).name
        if not filename.endswith('.mp4'):
            # If the URL path doesn't end with .mp4, assign a default name
            filename = "video.mp4"

    print(f"Downloading {video_url} to {filename}...")
    try:
        with requests.get(video_url, stream=True) as r:
            r.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
            with open(filename, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        print("Download complete!")
        return filename
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {video_url}: {e}")
        return None

In [ ]:

#@title ECのリスト操作

kind = "single_url" #@param ["init", "main", "spank", "single_url", "def"]
#@markdown <br>single_urlの場合、urlをurl_to_processに入れ、
#@markdown <br>そのままダウンロードする場合はdownloadをチェック
url_to_process = "https://zozovideo.com/171982/" #@param {type:"string"}
download = True #@param {type:'boolean'}
#@markdown <br>

import pathlib
import re
import requests
from bs4 import BeautifulSoup
import copy
import time
import random
import tkinter as tk
import json
import ast
import datetime

driver_list = {
  "init": True,
  "main": True,
  "spank": True,
  "single_url": True,
  "def": False
}

useDriver = driver_list[kind]

driver = None

if useDriver:
  initDriver()

siteList = {
  'zozovideo.com': {
    'name': 'zozo',
    'code': 0
  },
  'jp.spankbang.com': {
    'name': 'spank',
    'code': 1
  }
}

def getDate():
  return datetime.datetime.now().strftime('%Y/%m/%d-%H:%M')

def makeFile(path):
  if not path.exists():
    path.touch()


def getZozo(soup):
  result = {
    'title': 'Unknown',
    'status': [],
    'information': {

    }
  }
  #soup = BeautifulSoup(text, 'html.parser')
  video = soup.find(id='video')

  # 動画が見れない場合
  if not video:
    result['status'].append("Not found Video")
  else:
    poster = video['poster']
    video_src = video.find('source')['src']

    result['poster_url'] = poster
    result['video_url'] = video_src


  information = soup.find('div', {'class': 'information_box'})
  #print(information)

  # 作品情報が載っていない時
  if not information:
    result['status'].append("Not found Information")
  else:
    invalid = False
    for li in information.select('ul li'):
      key = li.select_one('.information-left')
      value = li.select_one('.information-right')
      if key and not value:
        tmp = BeautifulSoup(str(li), "html.parser").li
        k = tmp.select_one('.information-left')
        if k:
            k.decompose()
        value = tmp
      if not key or not value:
        if not invalid:
          result['status'].append('Invalid Information')
          result['information_code'] = str(information)
          invalid = True
          continue
      else:
        key_text = key.get_text(" ", strip=True).replace("：", "")
        value_text = value.get_text(" ", strip=True)
        result['information'][key_text] = value_text
        if key_text == 'タイトル':
          result['title'] = value_text
          result['series_url'] = value.find('a')['href']
  if not result['title']:
    result['title'] = soup.find('meta', attrs={'name': 'twitter:title'})
    if result['title']:
       result['title'] = result['title']['content']
    else:
      result['title'] = soup.title.text
      if not result['title']:
        result['title'] = 'No TITLE'

  return result

def getSpank(url, soup):
  result = {
    'title': 'Unknown',
    'status': [],
    'information': {

    }
  }

  def getUrl(result, soup):
    if soup.title.text == 'Just a moment...':
      print("Can't get code")
      print("Please try to 'spank' of kind")
      result['status'].append("Can't get code")
      return
    result['status'].append("Complete to get detail")
    main = soup.find('main')
    if not main:
      print("Not found 'main' tag")
      result['status'].append("Not found 'main' tag")
      return
    script = main.find('script')
    if not script:
      print("Not found 'script' tag")
      result['status'].append("Not found 'script' tag")
      return
    urls_re = re.search(r'var stream_data = ({[^\n]+})', script.prettify())
    if not urls_re:
      print("Can't get url")
      result['status'].append("Can't get url")
      return
    urls = urls_re.group(1)
    urls = ast.literal_eval(urls.strip())
    if not 'main' in urls.keys():
      print("Not found main url")
      result['status'].append("Not found main url")
      return
    result['video_url'] = urls['main'][0]

    result['poster_url'] = soup.find(id = 'player_cover_img').get('src')

  getUrl(result, soup)

  def getTitle(result, soup):
    # title
    video = soup.find(id='video')
    if not video:
      print("Not found 'video' of id")
      result['status'].append("Not found 'video' of id")
      return

    h1 = video.find('h1')
    if not h1:
      print("Not found 'h1' tag in 'video'")
      result['status'].append("Not found 'h1' tag in 'video'")
      return

    result['title'] = h1.text

  getTitle(result, soup)

  return result

def getBySelenium(url, screenshot=False, ms=1000):
  initDriver()
  driver.get(url)
  #time.sleep(ms/1000)
  if screenshot:
    w = driver.execute_script("return document.body.scrollWidth;")
    h = driver.execute_script("return document.body.scrollHeight;")

    # set window size
    driver.set_window_size(w,h)

    # Get Screen Shot
    driver.save_screenshot('/content/screenshoot.png')
  html = driver.page_source
  driver.quit()
  #time.sleep(random.uniform(2,4))
  return html

def getHtml(info):
  if info['site'] == 'zozo':
    return requests.get(info['url']).text
  elif info['site'] == 'spank':
    return getBySelenium(info['url'])

def accessAndGetInfo(info):
  global text
  if info['site'] == 'zozo':
    text = getHtml(info);
    soup = BeautifulSoup(text, 'html.parser')
    data = getZozo(soup)
    #print(data)
    return {
      'ok': True,
      'url': info['url'],
      'sitename': info['sitename'],
      'site': 'zozo',
      'data': data,
      'date': getDate()
    }
  elif info['site'] == 'spank':
    #text = getBySelenium(info['url'], 1000*random.uniform(1, 1.5))
    #soup = BeautifulSoup(text, 'html.parser')
    #data = getSpank(info['url'], soup)
    #取得が制限されるので一つずつ手動でgetBySeleniumやる！
    global spankList
    data = getSpank(info['url'], BeautifulSoup(getBySelenium(info['url'])))
    return {
      'ok': True,
      'url': info['url'],
      'sitename': info['sitename'],
      'site': 'spank',
      'data': data,
      'date': getDate()
    }
  return {
    'ok': False,
    'url': info['url'],
    'sitename': info['sitename'],
    'site': None,
    'data': {
      'status': ['Unknown site']
    },
    'date': datetime.date.today()
  }

def minProcess(url):
  url = url.strip()
  sitenameRe = r'^https?://([^/]+)'
  if not re.match(sitenameRe, url):
    return {
      'ok': False,
      'content': {
        'url': url,
        'status': ['不正なURLです。']
      }
    }
  else:
    sitename = re.match(sitenameRe, url).group(1)
    site = siteList.get(sitename)
    if not site:
      site = {
        'name': 'other',
        'code': 10
      }

    info = {
      'url': url,
      'sitename': sitename,
      'site': site['name']
    }

    return {
      'ok': True,
      'content': accessAndGetInfo(info),
      'site_code': site['code']
    }

def sort(info_list):
  return sorted(info_list, key=lambda x: (x['site_code'], x['status_code']))

def formatter(info):
  info = copy.deepcopy(info)
  Info = {}
#{'ok': True, 'content': {'ok': True, 'url': 'https://zozovideo.com/nariyuki-papakatsu-girls-episode-1/', 'sitename': 'zozovideo.com', 'site': 'zozo', 'data': {'title': 'なりゆき→パパ活GIRLS!! The Animation #1「J○×オジサマの快感セックスライフ♥」', 'status': [], 'information': {'タイトル': 'なりゆき→パパ活GIRLS!! The Animation #1「J○×オジサマの快感セックスライフ♥」', 'Title': 'Nariyuki Papakatsu Girls!! The Animation Episode 1', '発売日': '2018/02/23', '制作': 'Pink Pineapple'}, 'poster_url': 'https://zozovideo.com/wp-content/uploads/2021/04/359d1ea5136363cd5705ef49ab33d28a.jpg', 'video_url': 'https://hstorage.xyz/files/N/nariyuki-papakatsu-girls-animation/nariyuki-papakatsu-girls-animation-1.mp4', 'series_url': 'https://zozovideo.com/playlist/nariyuki-papakatsu-girls'}}}
  content = info['content']
  if not content:
    return None
  data = content['data']
  if not data:
    return None
  if not info['ok']:
    return {'site_code': 20, 'status_code': 20, 'status': data['status'], 'title': 'Unknown', 'url': {'origin': content['url']}, 'date': getDate()}
  else:
    if not content['ok']:
      return {'site_code': 20, 'status_code': 10, 'status': data['status'], 'title': 'Unknown', 'url': {'origin': content['url']}, 'date': getDate()}
    else:
      Info['site_code'] = info['site_code']
      if content['site'] == 'zozo':
        status = data['status']
        Info['status_code'] = 10
        Info['status'] = []
        if len(status) == 0:
          Info['status'] = ['成功']
          Info['status_code'] = 0
        elif "Not found Video" in status:
          Info['status'].append('動画が見つかりません')
          Info['status_code'] = 1
        elif "Not found Information" in status:
          Info['status'].append('作品情報が見つかりません')
          Info['status_code'] = 2
        elif 'Invalid Information' in status:
          Info['status'].append('作品情報が不正な型になっています。')
          Info['status_code'] = 3

        Info['title'] = data['title']
        Info['url'] = {
          'origin': content['url']
        }


        if 'video_url' in data.keys():
          Info['url']['video'] = data['video_url']
        if 'poster_url' in data.keys():
          Info['url']['poster'] = data['poster_url']
        if 'series_url' in data.keys():
          Info['url']['series_url'] = data['series_url']

        if 'information' in data.keys():
          Info['information'] = data['information']
        Info['date'] = content['date']
        return Info
      elif info['site_code'] == 1:
        status = data['status']
        Info['status_code'] = 20
        Info['status'] = []
        if 'Complete to get detail' in status:
          Info['status_code'] = 10
          if len(status) == 1:
            Info['status'].append('成功')
            Info['status_code'] = 0
          elif "Can't get code" in status:
            Info['status'].append('サイトにアクセスしてコードを取得できません')
            Info['status_code'] = 1
          elif "Not found 'main' tag" in status:
            Info['status'].append('mainタグが見つかりません')
            Info['status_code'] = 2
          elif "Not found 'script' tag" in status:
            Info['status'].append('scriptタグが見つかりません')
            Info['status_code'] = 2
          elif "Not found 'video' of id" in status:
            Info['status'].append('動画が見つかりません')
            Info['status_code'] = 3
          elif "Can't get url" in status:
            Info['status'].append('URLが見つかりません')
            Info['status_code'] = 4
          elif "Not found 'h1' tag in 'video'" in status:
            Info['status'].append('タイトルが見つかりません')
            Info['status_code'] = 5
          else:
            Info['status'].append('例外が発生しました')
            Info['status_code'] = 6
          Info['title'] = data['title']
          Info['url'] = {
            'origin': content['url']
          }

          if 'video_url' in data.keys():
            Info['url']['video'] = data['video_url']
          if 'poster_url' in data.keys():
            Info['url']['poster'] = data['poster_url']
          Info['date'] = content['date']
          return Info
        else:
          Info['status'].append('手動で詳細を取得してください。')
          Info['title'] = 'Unknown'
          Info['url'] = {
            'origin': content['url']
          }
          Info['date'] = content['date']
        return Info
      else:
        Info['status_code'] = 20
        Info['title'] = 'Unknown'
        Info['url'] = {
          'origin': content['url']
        }
        Info['date'] = content['date']
  return Info

def process(url_list):
  global spankList
  spankList = []
  print('start')
  info_list = []

  url_list_length = len(url_list)
  count = 1
  for url in url_list:
    print(str(count)+' / '+str(url_list_length))
    info_list.append(formatter(minProcess(url)))
    count += 1
  info_list = sort(info_list)
  for info in info_list:
    if info['site_code'] == 1:
      spankList.append(info['url']['origin'])
    print(info)
  print("Write in 'url-sort-list.txt'")
  makeFile(url_sort_list_path)
  with open(url_sort_list_path, 'w') as f:
    for info in info_list:
      f.write(info['title']+'\n - '+info['url']['origin']+'\n')
  #test(text, 'allCode')
  print("Write in 'spank-list.txt")
  makeFile(spank_list_path)
  with open(spank_list_path, 'w') as f:
    for info in info_list:
      if info['site_code'] == 1:
        f.write(str(info)+'\n')
  print("Write in 'info-list.txt'")
  makeFile(info_list_path)
  with open(info_list_path, 'w') as f:
    for info in info_list:
      f.write(str(info)+'\n')
  print('finish')

def init():
  if not folder_path.exists():
    raise Exception('folder not found')
  if not url_list_path.exists():
    raise Exception('list file not found')
  with open(url_list_path) as f:
    process(f.readlines())

def getOneSpank():
  global spankIndex
  url = spankList[spankIndex]
  getSpank(url, BeautifulSoup(getBySelenium(url)))
  spankIndex += 1
  if spankIndex == len(spankList):
    print("SpankList終了しました。")

if kind == 'init':
  init()
elif kind == 'single_url':
  if url_to_process:
    print(f"Processing single URL: {url_to_process}")
    result = formatter(minProcess(url_to_process))
    print(result)
    if download and 'url' in result and 'video' in result['url']:
      print(f"Attempting to download video from: {result['url']['video']}")
      download_mp4(result['url']['video'])
    elif download and ('url' not in result or 'video' not in result['url']):
      print("Download requested, but video URL not found in processed result.")
  else:
    print("Please provide a URL in 'url_to_process' for single_url mode.")

#while True:
#  input("Enter待ち")
#  getOneSpank()

#test(requests.get('https://zozovideo.com/48671/').text, 'information')

#print(requests.get('https://jp.spankbang.com/4492v/video/hime+sama+4').text)

#text = getBySelenium('https://jp.spankbang.com/3wlzl/video/kyonyuu+fantasy+2', ms=1120)

#test(getBySelenium('https://jp.spankbang.com/3wlzl/video/kyonyuu+fantasy+2'),'allCode')

#test(text, 'allCode')

#url = 'https://jp.spankbang.com/3wlzl/video/kyonyuu+fantasy+2'

def callSpank():
  for i in range(len(spankList)):
    print(str(i)+": "+spankList[i])

  while True:
    inp = input()
    if inp == '':
      return
    url = ""
    try:
      url = spankList[int(inp)]
    except:
      url = inp

    spankSoup = BeautifulSoup(getBySelenium(url))
    getSpank(url, spankSoup)

#callSpank()

"""if kind == "spank":
  with open("/content/drive/MyDrive/共有フォルダ/情報/ec/info-list.txt") as f:
    info_list = f.readlines()
  spank_info_list = []
  for ind in range(len(info_list)):
    info = ast.literal_eval(info_list[ind].strip())
    if info['series_code'] == 1:
      print(str(len(spank_info_list))+': '+str(info))
      spank_info_list.append(info)

  while True:
    info = input()
    if info == '':
      print('終了')
      break
    info = spank_info_list[int(info)]
    if not info:
      print('Error')
      continue
    if not info['status_code'] == 10:
      print('このURLから情報を取ることはできません。Statusを確認してください。')
      continue
    url = info['url']['origin']
    spankSoup = BeautifulSoup(getBySelenium(url))
    getSpank(url, spankSoup)
"""

def spank():
  spank_list_read = []
  with open(spank_list_path) as f:
    spank_list_read = f.readlines()
    spank_list = []
  for ind in range(len(spank_list_read)):
    spank_list.append(ast.literal_eval(spank_list_read[ind].strip()))
    print(str(ind)+': '+spank_list_read[ind].strip())

  while True:
    inp = input()
    if inp == '':
      print('終了')
      break
    ind = int(inp)
    info = spank_list[ind]
    if not info:
      print('Error')
      continue
    if not info['status_code'] == 20:
      print('このURLから情報を取ることはできません。Statusを確認してください。')
      continue
    url = info['url']['origin']
    spankSoup = BeautifulSoup(getBySelenium(url))
    info = getSpank(url, spankSoup)
    spank_list_read[ind] = str(info)+'\n'
    with open(spank_list_path, 'w') as f:
      f.writelines(spank_list_read)
    for i in range(len(spank_list)):
      print(str(i)+': '+spank_list_read[i].strip())

if kind == "spank":
  spank()

#text = getBySelenium('https://jp.spankbang.com/40lib/video/russian+milf+hentai+3')

#with open('/content/index.html', 'w') as f:
#  f.write(text)

#soup = BeautifulSoup(text, 'html.parser')
#print(soup.find_all('video'))

#RE = r'片田舎に嫁いできた'
#print(len(re.findall(RE, text)))
#for match in re.finditer(RE, text):
#  index = match.start()
#  print(text[index-500:index+500])
#  print('\n\n\n')

commentOut = """

--- ZOZO ---
 - 全て成功
{
  'status': '成功',
  'title': 'みだれうち ＃2',
  'sitename': 'zozovideo.com',
  'url': {
    'origin': 'https://zozovideo.com/midareuchi-episode-2/',
    'poster': 'https://zozovideo.com/wp-content/uploads/2024/04/a150372d66e68b79d99a4f3d87713a99.jpg',
    'video': 'https://hstorage.xyz/files/M/midareuchi/midareuchi-2_480p.mp4',
    'series': 'https://zozovideo.com/playlist/midareuchi/'
  },
  'information': {
    'タイトル': 'みだれうち ＃2',
    'Title': 'Midareuchi Episode 2',
    '発売日': '2024/04/26',
    '制作': 'あんてきぬすっ'
  },
}
 - 動画の取得に失敗
{
  'status': '動画の取得に失敗',
  'title': 'みだれうち ＃2',
  'sitename': 'zozovideo.com',
  'url': {
    'origin': 'https://zozovideo.com/midareuchi-episode-2/',
    'poster': 'https://zozovideo.com/wp-content/uploads/2024/04/a150372d66e68b79d99a4f3d87713a99.jpg',
    'video': 'https://hstorage.xyz/files/M/midareuchi/midareuchi-2_480p.mp4',
    'series': 'https://zozovideo.com/playlist/midareuchi/'
  },
  'information': {
    'タイトル': 'みだれうち ＃2',
    'Title': 'Midareuchi Episode 2',
    '発売日': '2024/04/26',
    '制作': 'あんてきぬすっ'
  },
}

--- spankbang ---
 - 詳細を取得前
{
  'ok': True,
  'content': {
    'ok': True,
    'url': info['url'],
    'sitename': info['sitename'],
    'site': 'spank',
    'data': {
      'status': ['Wait to get detail']
    },
    'date': getDate()
  },
  'site_code': 1
}
<main class="main-container" data-testid="main">
   <!-- prettier-ignore-start -->
   <script type="text/javascript">
    var ana_video_id = '6560337';
        var stream_data = {'240p': ['https://vdownload-5.sb-cd.com/6/5/6560337-240p.mp4?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337'], '320p': [], '480p': ['https://vdownload-5.sb-cd.com/6/5/6560337-480p.mp4?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337'], '720p': ['https://vdownload-5.sb-cd.com/6/5/6560337-720p.mp4?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337'], '1080p': [], '4k': [], 'mpd': [], 'm3u8': ['https://hls-uranus.sb-cd.com/hls/6/5/6560337-,240p,480p,720p,.mp4.urlset/master.m3u8?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337'], 'm3u8_240p': ['https://hls-uranus.sb-cd.com/hls/6/5/6560337-,240p,.mp4.urlset/master.m3u8?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337'], 'm3u8_320p': [], 'm3u8_480p': ['https://hls-uranus.sb-cd.com/hls/6/5/6560337-,480p,.mp4.urlset/master.m3u8?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337'], 'm3u8_720p': ['https://hls-uranus.sb-cd.com/hls/6/5/6560337-,720p,.mp4.urlset/master.m3u8?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337'], 'm3u8_1080p': [], 'm3u8_4k': [], 'cover_image': 'https://tbi.sb-cd.com/t/6560337/db/b8/w:1280/t6-enh/kyonyuu-fantasy-2.jpg', 'thumbnail': 'https://tbi.sb-cd.com/t/6560337/db/b8/w:300/t6-enh/kyonyuu-fantasy-2.jpg', 'stream_raw_id': 6560337, 'stream_sheet': 'https://tbv.sb-cd.com/t/6560337/db/b8/t9999.jpg', 'length': 1790, 'main': ['https://vdownload-5.sb-cd.com/6/5/6560337-720p.mp4?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337']};
        var live_keywords = 'hentai,big tits,cartoon,stripchat,demon,kyonyuu fantasy,';



            var ads_channel = 'UGC';
   </script>
   <script type="text/javascript">
    window.dataLayer.push({
            'recommendation_scenario': "2",
        });
   </script>
   <script type="text/javascript">
    window.next_video = ' /83t0u/video/todo+lo+que+quiero+hacer+es+que+me+llenen+la+playa ';
   </script>
   <!-- prettier-ignore-end -->
   <div id="video_theater">
   </div>
   <div class="main_content_container">
    <div data-streamkey="NjU2MDMzNw.jH_RQO9MwkrvNyjI3-FMXMsuBAQ" data-testid="video-page" data-videoid="6560337" id="video" x-data="videoPage">
     <div class="left flex flex-col gap-2">
      <div class="flex flex-row gap-4">
       <h1 class="main_content_title" title="巨乳ファンタジー2">
        巨乳ファンタジー2
       </h1>
       <span class="responsive-page views">
        <svg class="i_svg i_new-ui-visible-outlined">
         <use xlink:href="#new-ui-visible-outlined">
         </use>
        </svg>
        94,945 ビュー
       </span>
      </div>
      <div class="relative z-0" x-data="horizontalScroller('', '', undefined)">
       <span @click="scrollLeft()" @mousedown.prevent="if ($event.detail &gt; 1) $event.preventDefault();" class="before:from-kobalt-700 absolute top-0 -left-1 z-10 flex h-full cursor-pointer items-center before:pointer-events-none before:absolute before:top-0 before:left-0 before:h-full before:w-24 before:bg-gradient-to-r" style="display: none;" x-show="showPrev" x-transition="">
        <svg class="i_svg i_chevron-left">
         <use xlink:href="#chevron-left">
         </use>

"""

Processing single URL: https://zozovideo.com/171982/
{'site_code': 0, 'status_code': 0, 'status': ['成功'], 'title': '祓魔少女シャルロット 第一話 聖衣、着装！エクソシスト、シャルロットが悪霊を祓う！', 'url': {'origin': 'https://zozovideo.com/171982/', 'video': 'https://na-01.javprovider.com/F/futsuma-shoujo-charlotte-1.mp4', 'poster': 'https://zozovideo.com/wp-content/uploads/2025/04/90185c049f494c1029521b53f65e9659.jpg', 'series_url': 'https://zozovideo.com/playlist/futsuma-shoujo-charlotte/'}, 'information': {'タイトル': '祓魔少女シャルロット 第一話 聖衣、着装！エクソシスト、シャルロットが悪霊を祓う！', 'Title': 'Futsuma Shoujo Charlotte Episode 1', '発売日': '2024/09/27', '制作': '魔人'}, 'date': '2026/05/30-01:40'}
Attempting to download video from: https://na-01.javprovider.com/F/futsuma-shoujo-charlotte-1.mp4


In [ ]:

#@title VIEWER

import pathlib
import ast
from IPython.display import HTML

def drawHtml():
  with open(info_list_path, 'r') as f:
    info_list = f.readlines()

  html = '<input type="button" onclick="toggle()" value="Button"><div id="container" style="display: none;">'
  for ind in range(len(info_list)):
    info = ast.literal_eval(info_list[ind].strip())
    if 'url' in info.keys():
      url = ''
      if info['site_code'] == 1:
        if 'video' in info['url'].keys():
          html += '<a href="'+info['url']['video']+'">VIDEO<br></a>'
        url = info['url']['origin']
      else:
        if 'video' in info['url'].keys():
          url = info['url']['video']
        else:
          url = info['url']['origin']
      html += '<a href="'+url+'">'
      html += info['title']+"<br>"
      if 'poster' in info['url'].keys():
        poster = info['url']['poster']
        html += '<img src="'+poster+'" style="width: 80px;">'
        #print(poster)
      html += '</a><br>'
  html += """</div><script>view = false;function toggle(){
    document.getElementById("container").style.display = !view ? "block" : "none";
    view = !view;
  }"""
  with open(folder_path / 'viewer.html', 'w') as f:
    f.write(html)
  return HTML(html)

drawHtml()

In [ ]:

#@title 動画をURLからダウンロード

import requests
from urllib.parse import urlparse
import pathlib

download_mp4()

video_url_from_user = 'https://jp.spankbang.com/3wlzl/video/kyonyuu+fantasy+2' #@param{type: "string"}
downloaded_file = download_mp4(video_url_from_user, filename='downloaded_from_user.mp4')
if downloaded_file:
    print(f"Video saved as: {downloaded_file}")
else:
    print("Video download failed.")

Error downloading https://jp.spankbang.com/3wlzl/video/kyonyuu+fantasy+2: 403 Client Error: Forbidden for url: https://jp.spankbang.com/3wlzl/video/kyonyuu+fantasy+2
Video download failed.


In [ ]:
!pip install yt_dlp
import yt_dlp

#@title Get from site: 2
url = "https://jp.spankbang.com/591u9/video/ryou+seibai+gakuen" #@param{type: "string"}
download_method = "spank" #@param ["minimum", "spank"]


if download_method == "minimum":
  ydl_opts = {}

  with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])

elif download_method == "spank":
  from urllib.parse import urlparse
  # Configure ydl_opts with general headers

  url_to_process = url
  ydl_opts = {
    'http_headers': {
      'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Safari/537.36',
      'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
      'Accept-Language': 'en-us,en;q=0.5',
      'Sec-Fetch-Mode': 'navigate',
    },
    'cookiefile': None, # If cookies are needed, this can be set to a file path.
    'noplaylist': True, # To ensure only the single video is downloaded, not a playlist.
  }

  video_final_url = None

  if download_method in ['spank', 'zozo']:
    print(f"Attempting to extract video URL for '{url_to_process}' using Selenium for {download_method} site...")
    # We need to create a dummy info dictionary for accessAndGetInfo
    dummy_info = {
      'url': url_to_process,
      'sitename': 'jp.spankbang.com' if download_method == 'spank' else 'zozovideo.com', # Must match one of the keys in siteList
      'site': download_method # Must match one of the 'name' in siteList values
    }

    # Call the function that uses Selenium to extract video info
    extracted_info = accessAndGetInfo(dummy_info)

    if extracted_info and extracted_info['ok'] and 'video_url' in extracted_info['data']:
      video_final_url = extracted_info['data']['video_url']
      print(f"Extracted video URL: {video_final_url}")
      # Set referer to the original page for the extracted video URL download
      ydl_opts['http_headers']['Referer'] = url_to_process
    else:
      print(f"Failed to extract video URL for {url_to_process}. Status: {extracted_info['data']['status'] if extracted_info and 'data' in extracted_info else 'Unknown'}")
      print("No direct video URL found after extraction.")

  elif download_method in ['porn', 'other']:
    print(f"Attempting direct download using yt-dlp for '{url_to_process}' for {download_method} site.")
    video_final_url = url_to_process
    # For direct download, the referer should match the target URL's domain
    parsed_url = urlparse(url_to_process)
    ydl_opts['http_headers']['Referer'] = f"{parsed_url.scheme}://{parsed_url.netloc}/"

  else:
    print(f"Unknown download method: {download_method}. Please select from 'porn', 'spank', 'zozo', 'other'.")


  if video_final_url:
    try:
      with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([video_final_url])
    except Exception as e:
      print(f"An error occurred during yt-dlp download: {e}")
  else:
    print("Download cannot proceed: No valid video URL to process.")


In [ ]:


#@title 全文取得
kind1 = "Driver" #@param ["Driver", "Requests"]
kind_test = "Search" #@param ["Code", "Search"]
#@markdown <br>
url = "https://jp.spankbang.com/40lib/video/russian+milf+hentai+3" #@param{type: "string"}
#@markdown kind_test == "Search"の時の変数
search_word = "ロシア" #@param{type: "string"}

#https://jp.pornhub.com/view_video.php?viewkey=648949cfe2d34

html = ""
if kind1 == "Driver":
  initDriver()
  driver.get(url)
  html = driver.page_source
  driver.quit()
elif kind1 == "Requests":
  html = requests.get(url).text

if kind_test == "Search":
  print(html)
  test(html, 'search', {'re': search_word, 'range': 500})
elif kind_test == "Code":
  print(html)

ストリーミング出力は最後の 5000 行に切り捨てられました。

            

            <div class="text-body-sm text-primary absolute right-2 bottom-2 rounded bg-neutral-900/75 px-1" data-testid="video-item-length">
                11m
            </div>
        
    </a>


        
            
    

    <div x-data="videoInfo" data-testid="video-info-with-badge" class="responsive-page mt-1.5">
        <div class="flex justify-between">
            <span class="flex min-w-0 items-center">
                <span data-testid="badge" data-badge="tag" class="mr-0.5 inline-flex text-icon-2xs text-icon-entity md:mr-1">
        
            <svg class="i_svg i_icon-hashtag"><use xlink:href="#icon-hashtag"></use></svg>
        
    </span>
                <a data-testid="title" class="!inline truncate" href="/s/hentai/">
    
    <span class="text-action-tertiary text-body-sm  md:text-body-md">Hentai</span>

</a>
            </span>
            <span class="ml-0.5 flex items-center justify-end whitespace-nowrap text-body